# LARES — ACSAC 2026 Artifact Evaluation (Google Colab)

**Paper:** *LARES: Host-centered Lateral Movement Detection via Inductive Graph Reasoning*

Run every cell top to bottom. The notebook installs the environment, fetches the released
weights and a small data bundle, runs the commands documented in the repository
`README.md`, and prints the reproduced numbers next to the published ones.

- **Runtime:** about 20 minutes.
- **Download:** about 0.3 GB for LANL, a few GB with OpTC.
- **Hardware:** a CPU runtime is enough. A T4 GPU is faster but not required.

## Claims reproduced here

1. **Precision under class imbalance (§V-B, Figure 2, Table II).** 70% edge-level precision
   on LANL with 104 false positives, against 1,408 for ARGUS and 9,286 for EULER.
2. **Inductive detection (§V-C, Table II, Exp1-Exp3).** Detection holds when 30% and 50% of
   hosts, including all malicious ones, are unseen during training.
3. **Source host localization (§E, Table VII).** One false positive among 13.2k LANL hosts,
   and 2 of 3 root attack hosts on OpTC with a single false positive.
4. **Two-stage detection (§V-E, Table III, *Detection* row).** Replacing it with direct edge
   thresholding collapses precision.

These are evaluated from the released weights, on the compiled test snapshots the paper was
evaluated on. Everything upstream of that, downloading the 14.35 GB preprocessed dataset,
compiling snapshots, and retraining, lives in the companion notebook
`colab/LARES_full_pipeline.ipynb`. Reviewers do not need it.

---
## 1. Environment check

In [ ]:
import os, platform, subprocess, sys

print('Python :', sys.version.split()[0])
print('OS     :', platform.platform())
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                   '--format=csv,noheader']).decode().strip())
except Exception:
    print('GPU    : none detected (CPU runtime; Path A still works)')
print()
mount = '/content' if os.path.isdir('/content') else '/'
print(subprocess.check_output(['df', '-h', mount]).decode())

---
## 2. Configuration

Everything the notebook needs is set here. The two bundle URLs point to the lightweight
archives produced by `tools/make_colab_bundle.py`; the full dataset URL is the MEGA link
given in the paper.

In [ ]:
# --- Source code -----------------------------------------------------------------
# The notebook tries, in order: an existing checkout at REPO_DIR, REPO_GIT_URL,
# REPO_ARCHIVE_URL, then an interactive upload. Leave a field empty to skip it.
REPO_GIT_URL     = 'https://github.com/TristanBilot/nids.git'
REPO_ARCHIVE_URL = ''   # a .zip/.tar.gz of this repository: https, mega.nz, Drive, or a local path
REPO_DIR         = '/content/lares'
DOWNLOAD_DIR     = '/content/downloads'
#
# For the anonymous artifact submission, replace REPO_GIT_URL with an anonymised mirror,
# since the URL above identifies the authors. Mirrors on anonymous.4open.science expire
# after a few months; the API then answers 410 {"error": "repository_expired"}, which is
# what the link in footnote 1 of the paper currently returns. Point REPO_ARCHIVE_URL at a
# fresh one instead:
#   REPO_GIT_URL     = ''
#   REPO_ARCHIVE_URL = 'https://anonymous.4open.science/api/repo/<new-id>/zip'

# --- Data ------------------------------------------------------------------------
DATA_ROOT = '/content/lanl_optc_datasets'   # exported as LARES_DATA_ROOT

# Path A: lightweight bundles (compiled test snapshots only).
# Replace with the public URLs of the archives built by tools/make_colab_bundle.py.
# Supported: direct https, mega.nz, or Google Drive links.
BUNDLE_URLS = {
    'LANL': '',   # e.g. 'https://mega.nz/file/XXXXXXXX#YYYYYYYYYYYYYYYY'
    'OPTC': '',
}

# Optional. MEGA rate-limits anonymous downloads per IP address; set your account here to
# download the bundle through it instead. The password is asked interactively and written
# only to ~/.megarc inside this runtime.
MEGA_USERNAME = ''

# --- What to run -----------------------------------------------------------------
DATASETS    = ['LANL']        # ['LANL', 'OPTC'] once the OpTC bundle is available
EXPERIMENTS = [0, 1, 2, 3]    # Exp0 transductive, Exp1-Exp3 inductive

os.environ['LARES_DATA_ROOT'] = DATA_ROOT
os.makedirs(DATA_ROOT, exist_ok=True)
print('Data root:', DATA_ROOT)

---
## 3. Download helpers

Handles direct HTTPS links, MEGA links and Google Drive links.

MEGA links cannot be fetched with `curl` or `wget`: the file is end-to-end encrypted and
the decryption key lives in the URL fragment after `#`, which is never sent to the server.
`megadl` from `megatools` performs the API call, the transfer and the decryption, so it is
installed on demand here.

> **MEGA quota warning.** Anonymous MEGA links are bandwidth-limited per IP address, and
> Colab exits through shared addresses that are often already over quota. The small
> bundles of section 6 download fine; the 14.35 GB archive frequently aborts. Set
> `MEGA_USERNAME` in section 2 to draw on your own account's quota instead, or mount a
> Google Drive copy and point the URL at that path:
>
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> FULL_DATASET_URL = '/content/drive/MyDrive/lanl_optc_datasets.zip'
> ```

In [ ]:
import glob, os, shutil, subprocess

def sh(cmd, check=True):
    """Run a shell command, streaming its output into the notebook.

    subprocess writes to the kernel's file descriptors, which Colab does not show in the
    cell, so output is piped back and printed here. Without this a failing command raises
    with no visible reason.
    """
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end='', flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f'command failed with exit code {code}: {cmd}')
    return code

def _install_megatools():
    if shutil.which('megadl') is None:
        sh('apt-get -qq update', check=False)
        sh('apt-get -qq install -y megatools', check=False)
    return shutil.which('megadl') is not None

def _mega_login():
    """Store MEGA credentials in ~/.megarc so megadl uses the account's quota.

    megatools reads this file on its own, which keeps the password off the command line.
    """
    if not MEGA_USERNAME:
        return
    rc = os.path.expanduser('~/.megarc')
    if os.path.isfile(rc):
        return
    import getpass
    pw = getpass.getpass(f'MEGA password for {MEGA_USERNAME} (not echoed, not logged): ')
    with open(rc, 'w') as f:
        f.write(f'[Login]\nUsername = {MEGA_USERNAME}\nPassword = {pw}\n')
    os.chmod(rc, 0o600)
    print('Credentials written to ~/.megarc')

def download(url, dest_dir=None):
    """Download `url` into `dest_dir` and return the path of the downloaded file."""
    dest_dir = dest_dir or DOWNLOAD_DIR
    if url.startswith('/') or url.startswith('file://'):
        return url.replace('file://', '')

    os.makedirs(dest_dir, exist_ok=True)

    if 'mega.nz' in url:
        if not _install_megatools():
            raise RuntimeError('megatools could not be installed; use a direct URL, '
                               'or a Google Drive copy of the archive')
        _mega_login()
        before = set(glob.glob(os.path.join(dest_dir, '*')))
        sh(f'megadl --path {dest_dir} "{url}"')
        new = set(glob.glob(os.path.join(dest_dir, '*'))) - before
        if not new:
            raise RuntimeError(
                'megadl produced no file. The link is most likely over its per-IP '
                'bandwidth quota: set MEGA_USERNAME in section 2 to download through '
                'your own account, or copy the archive to Google Drive, mount it with '
                "google.colab.drive, and point the URL at that local path.")
        return new.pop()

    if 'drive.google.com' in url:
        sh('pip -q install -U gdown', check=False)
        import gdown
        return gdown.download(url=url, output=dest_dir + '/', fuzzy=True, quiet=False)

    name = url.split('/')[-1].split('?')[0] or 'download.bin'
    out = os.path.join(dest_dir, name)
    sh(f'wget -q --show-progress -c -O "{out}" "{url}"')
    return out

def extract(archive, dest):
    """Extract a .tar.gz / .tgz / .tar / .zip archive into `dest`."""
    os.makedirs(dest, exist_ok=True)
    if archive.endswith(('.tar.gz', '.tgz', '.tar')):
        sh(f'tar -xf "{archive}" -C "{dest}"')
    elif archive.endswith('.zip'):
        sh(f'unzip -q -o "{archive}" -d "{dest}"')
    else:
        raise ValueError(f'unknown archive type: {archive}')
    return dest

print('helpers ready')

---
## 4. Fetch the code

In [ ]:
import os, shutil

def find_repo_root(base, max_depth=3):
    """Locate the folder holding src/main.py inside an extracted archive."""
    base = os.path.abspath(base)
    for dirpath, dirnames, _ in os.walk(base):
        dirnames[:] = [d for d in dirnames if d != '__MACOSX']
        if dirpath[len(base):].count(os.sep) > max_depth:
            dirnames[:] = []
            continue
        if os.path.isfile(os.path.join(dirpath, 'src', 'main.py')):
            return dirpath
    return None

def install_repo_from_archive(archive):
    staging = REPO_DIR.rstrip('/') + '_staging'
    shutil.rmtree(staging, ignore_errors=True)
    extract(archive, staging)
    root = find_repo_root(staging)
    if root is None:
        raise RuntimeError(f'{archive} does not contain a src/main.py')
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    shutil.move(root, REPO_DIR)

if os.path.isfile(os.path.join(REPO_DIR, 'src', 'main.py')):
    print('Repository already present at', REPO_DIR)
elif REPO_GIT_URL:
    sh(f'git clone --depth 1 {REPO_GIT_URL} {REPO_DIR}')
elif REPO_ARCHIVE_URL:
    install_repo_from_archive(download(REPO_ARCHIVE_URL))
else:
    # Last resort: upload a zip of the repository from your machine.
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Set REPO_GIT_URL or REPO_ARCHIVE_URL in section 2.')
    print('No code source configured. Upload a .zip or .tar.gz of the repository '
          '(sources + weights/, about 60 MB):')
    uploaded = files.upload()
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    name = next(iter(uploaded))
    path = os.path.join(DOWNLOAD_DIR, name)
    with open(path, 'wb') as f:
        f.write(uploaded[name])
    install_repo_from_archive(path)

os.chdir(REPO_DIR)
assert os.path.isfile('src/main.py'), 'src/main.py not found'

missing = [f'weights_{d}_inductive_exp{e}.pkl'
           for d in DATASETS for e in EXPERIMENTS
           if not os.path.isfile(f'weights/weights_{d}_inductive_exp{e}.pkl')]
print('Repository:', REPO_DIR)
if missing:
    print('WARNING: missing weight files, Path A cannot run:', missing)
else:
    print('All weights required by DATASETS/EXPERIMENTS are present.')
sh('ls -l weights | head', check=False)

---
## 5. Install dependencies

Colab already ships PyTorch, pandas, scikit-learn, joblib and tqdm, so only PyTorch
Geometric and Weights & Biases are added. The compiled PyG extensions
(`torch_scatter`, `torch_sparse`, `torch_cluster`) listed in the README are **not**
required: no module in `src/` imports them, and installing them from source on Colab
takes tens of minutes.

In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)

sh('pip -q install torch_geometric==2.6.1 wandb==0.17.9', check=False)

import torch_geometric
print('torch_geometric:', torch_geometric.__version__)

# Weights & Biases is imported by src/main.py but disabled at runtime.
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_SILENT'] = 'true'

---
## 6. Get the data

Downloads one bundle per dataset in `DATASETS`. A bundle holds only the **compiled test
snapshots**, which is all `--use_weights=True` reads: the training and validation loaders
are constructed but never iterated, and the inductive masking of Exp0-Exp3 applies to the
*training* split only, so the test split is identical across the four experiments. The
bundle stores it once and symlinks the four experiment names to it.

In [ ]:
import glob, os

def snapshots_of(dataset, exp):
    return glob.glob(os.path.join(DATA_ROOT, dataset, 'compiled', 'test',
                                  f'{dataset}_exp{exp}', '*.pkl'))

def missing_experiments():
    return [(ds, e) for ds in DATASETS for e in EXPERIMENTS if not snapshots_of(ds, e)]

def fetch_bundle(dataset):
    url = BUNDLE_URLS.get(dataset, '')
    if not url:
        raise SystemExit(
            f'No data for {dataset}.\n\n'
            f'Either set BUNDLE_URLS["{dataset}"] in section 2 to a bundle built with\n'
            f'    export LARES_DATA_ROOT=/path/to/lanl_optc_datasets\n'
            f'    python tools/make_colab_bundle.py --dataset {dataset} --out ./bundles\n'
            f'on a machine that already holds the compiled dataset, or build it in\n'
            f'Colab with the companion notebook colab/LARES_full_pipeline.ipynb.')
    extract(download(url), DATA_ROOT)

todo = missing_experiments()
if todo:
    for ds in sorted({ds for ds, _ in todo}):
        fetch_bundle(ds)
    still = missing_experiments()
    assert not still, f'the bundles are missing compiled snapshots for {still}'
    print('Bundles installed.')
else:
    print('All requested experiments already have compiled test snapshots.')

for ds in DATASETS:
    print(f'{ds}: ' + ', '.join(f'Exp{e}={len(snapshots_of(ds, e))} snapshots'
                                for e in EXPERIMENTS))

---
## 7. Run the evaluation

These are the exact commands of the README, section *Reproduce experiments → From weights*:

```shell
python src/main.py --config=LANL_inductive_exp0 --use_weights=True
...
python src/main.py --config=OPTC_inductive_exp3 --use_weights=True
```

Each run prints a node-level block (stage 1 of the detection, Table VII) followed by an
edge-level block (stage 2, Table II).

In [ ]:
import re, subprocess, sys, time

LOG_DIR = '/content/run_logs'
os.makedirs(LOG_DIR, exist_ok=True)

def run_config(config, extra_args='', tag=None):
    """Run one evaluation and return (stdout, wall_clock_seconds)."""
    tag = tag or config
    cmd = f'python src/main.py --config={config} --use_weights=True {extra_args}'.strip()
    print('$', cmd, flush=True)
    start = time.time()
    proc = subprocess.run(cmd, shell=True, cwd=REPO_DIR, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = time.time() - start
    with open(os.path.join(LOG_DIR, f'{tag}.log'), 'w') as f:
        f.write(proc.stdout)
    if proc.returncode != 0:
        print(proc.stdout[-4000:])
        raise RuntimeError(f'{cmd} failed with code {proc.returncode}')
    print(f'  done in {elapsed:.0f}s -> {LOG_DIR}/{tag}.log', flush=True)
    return proc.stdout, elapsed

NUM = r'(-?\d+\.?\d*|nan)'

def parse_block(block):
    """Extract the metrics printed by src/utils/eval.py for one detection level."""
    m = re.search(rf'AP: {NUM}.*?recall: {NUM} \| precision: {NUM} \| MCC: {NUM}', block, re.S)
    c = re.search(rf'TP: {NUM}/(\d+) \| FP: {NUM}/(\d+)', block)
    if not (m and c):
        return None
    tp, pos, fp, neg = float(c.group(1)), int(c.group(2)), float(c.group(3)), int(c.group(4))
    return {'TP': int(tp), 'FP': int(fp), 'FN': pos - int(tp), 'TN': neg - int(fp),
            'Recall': float(m.group(2)), 'Precision': float(m.group(3)),
            'MCC': float(m.group(4)), 'AP': float(m.group(1))}

def parse_output(stdout):
    """Split the run output into its node-level and edge-level metric blocks."""
    out = {}
    for key, header in [('node', 'Node detection metrics:'),
                        ('edge', 'Edge detection metrics:'),
                        ('edge', 'Standard edge-level detection:')]:
        if header in stdout:
            block = stdout.split(header)[-1]
            parsed = parse_block(block[:block.find('\n\n\n') if '\n\n\n' in block else 600])
            if parsed:
                out[key] = parsed
    return out

print('runner ready')

In [ ]:
results = {}

for ds in DATASETS:
    for e in EXPERIMENTS:
        config = f'{ds}_inductive_exp{e}'
        stdout, elapsed = run_config(config)
        parsed = parse_output(stdout)
        parsed['elapsed_s'] = elapsed
        results[(ds, f'Exp{e}')] = parsed

print('\nAll runs finished.')

---
## 7.1 Comparison with the paper

Reference values are transcribed from Table II (edge level) and Table VII (node level).
The `Δ` columns show the difference between this run and the paper. Small deviations are
expected across library and hardware versions; the claims are about orders of magnitude
in false positives, so differences of a few units are not meaningful.

In [ ]:
import pandas as pd

# Table II, LARES rows (paper).
PAPER_EDGE = {
    ('LANL', 'Exp0'): dict(TP=247, FP=104, Recall=0.61, Precision=0.70, MCC=0.65, AP=0.70),
    ('LANL', 'Exp1'): dict(TP=241, FP=116, Recall=0.59, Precision=0.68, MCC=0.63, AP=0.66),
    ('LANL', 'Exp2'): dict(TP=241, FP=116, Recall=0.59, Precision=0.68, MCC=0.63, AP=0.66),
    ('LANL', 'Exp3'): dict(TP=241, FP=156, Recall=0.59, Precision=0.61, MCC=0.60, AP=0.62),
    ('OPTC', 'Exp0'): dict(TP=27,  FP=63,  Recall=0.46, Precision=0.30, MCC=0.37, AP=0.39),
    ('OPTC', 'Exp1'): dict(TP=27,  FP=63,  Recall=0.46, Precision=0.30, MCC=0.37, AP=0.39),
    ('OPTC', 'Exp2'): dict(TP=24,  FP=66,  Recall=0.41, Precision=0.27, MCC=0.33, AP=0.37),
    ('OPTC', 'Exp3'): dict(TP=27,  FP=63,  Recall=0.49, Precision=0.30, MCC=0.37, AP=0.36),
}

# Table VII, LARES rows (paper); identical across Exp0-Exp3.
PAPER_NODE = {
    'LANL': dict(TP=1, FP=1, Recall=1.00, Precision=0.50, MCC=0.71),
    'OPTC': dict(TP=2, FP=1, Recall=0.67, Precision=0.67, MCC=0.67),
}

rows = []
for (ds, exp), res in results.items():
    got, ref = res.get('edge', {}), PAPER_EDGE[(ds, exp)]
    rows.append({
        'Dataset': ds, 'Exp': exp,
        'TP': got.get('TP'), 'TP (paper)': ref['TP'],
        'FP': got.get('FP'), 'FP (paper)': ref['FP'],
        'Precision': got.get('Precision'), 'Precision (paper)': ref['Precision'],
        'Recall': got.get('Recall'), 'Recall (paper)': ref['Recall'],
        'MCC': got.get('MCC'), 'MCC (paper)': ref['MCC'],
        'Runtime (s)': round(res.get('elapsed_s', float('nan'))),
    })
edge_df = pd.DataFrame(rows)

rows = []
for (ds, exp), res in results.items():
    got, ref = res.get('node', {}), PAPER_NODE[ds]
    rows.append({
        'Dataset': ds, 'Exp': exp,
        'TP': got.get('TP'), 'TP (paper)': ref['TP'],
        'FP': got.get('FP'), 'FP (paper)': ref['FP'],
        'Precision': got.get('Precision'), 'Precision (paper)': ref['Precision'],
        'MCC': got.get('MCC'), 'MCC (paper)': ref['MCC'],
    })
node_df = pd.DataFrame(rows)

print('Edge-level detection - paper Table II (LARES rows)')
display(edge_df)
print('\nNode-level source host detection - paper Table VII (LARES rows)')
display(node_df)

edge_df.to_csv('/content/lares_table2_reproduction.csv', index=False)
node_df.to_csv('/content/lares_table7_reproduction.csv', index=False)
print('\nSaved to /content/lares_table2_reproduction.csv and /content/lares_table7_reproduction.csv')

---
## 7.2 False positives against the baselines (Figure 2)

The baselines are separate code bases and are not rerun here. Their published counts on
LANL in the transductive setting (Exp0) are used as the comparison point, exactly as in
Figure 2 of the paper.

In [ ]:
baseline_fp = {'EULER': (297, 9286), 'ARGUS': (269, 1408), 'JBEIL': (89, 31067)}

lares = results.get(('LANL', 'Exp0'), {}).get('edge')
if lares is None:
    print('Run LANL Exp0 first (section 7).')
else:
    rows = [{'System': k, 'TP': tp, 'FP': fp, 'FP per TP': round(fp / tp, 1)}
            for k, (tp, fp) in baseline_fp.items()]
    rows.append({'System': 'LARES (this run)', 'TP': lares['TP'], 'FP': lares['FP'],
                 'FP per TP': round(lares['FP'] / max(lares['TP'], 1), 1)})
    df = pd.DataFrame(rows)
    df['FP reduction vs LARES'] = (df['FP'] / lares['FP']).round(1).astype(str) + 'x'
    display(df)

---
## 7.3 Two-stage detection ablation (Table III, *Detection* row)

Passing `--use_direct_edge_detection=True` replaces the two-stage procedure with the
direct edge thresholding used by EULER and ARGUS, at a fixed recall. This is the
ablation supporting limitation (2) of §II-B, and it needs no retraining.

In [ ]:
ablation = []
for ds in DATASETS:
    config = f'{ds}_inductive_exp0'
    stdout, _ = run_config(config, '--use_direct_edge_detection=True',
                           tag=f'{config}_direct_edge')
    direct = parse_output(stdout).get('edge', {})
    two_stage = results[(ds, 'Exp0')]['edge']
    ablation += [
        {'Dataset': ds, 'Detection': 'two-stage (LARES)', **{k: two_stage[k]
         for k in ['TP', 'FP', 'Precision', 'Recall', 'MCC']}},
        {'Dataset': ds, 'Detection': 'direct edge thresholding', **{k: direct.get(k)
         for k in ['TP', 'FP', 'Precision', 'Recall', 'MCC']}},
    ]
display(pd.DataFrame(ablation))

---
## 8. Mapping to the paper, and known limitations

| Paper item | Reproduced by |
|---|---|
| Table II, LARES rows | section 7.1 |
| Figure 2, FP counts on LANL | section 7.2 |
| Table VII, LARES rows | section 7.1 |
| Table III, *Detection* row | section 7.3 |
| Table III, encoder / decoder / feature rows | `colab/LARES_full_pipeline.ipynb`, then the README *Ablation study* commands |
| Figures 1 and 4, unseen-host sweeps | `colab/LARES_full_pipeline.ipynb`, then the README *MCC @ 10-100% of unseen hosts* commands |
| Figures 6 and 7, poisoning and evasion | requires retraining on perturbed graphs |
| Figures 9 and 10, runtime and memory | requires the baselines' own repositories |
| Figure 12, ACSAC'25 enterprise dataset | third-party dataset, not redistributed here |

**Limitations.** Baseline systems (EULER, ARGUS, JBEIL) are not rerun; their numbers come
from the paper and from the original repositories cited in §IV-B. This notebook evaluates
released weights, so it validates detection quality rather than training reproducibility.

## Troubleshooting

| Symptom | Cause and fix |
|---|---|
| `megadl` produces no file | MEGA per-IP quota reached. Set `MEGA_USERNAME` in section 2, or mount a Drive copy of the bundle and point `BUNDLE_URLS` at that path. |
| `No data for LANL` | `BUNDLE_URLS` is empty. See the message printed by section 6. |
| `FileNotFoundError: Neither compiled snapshots ... nor raw 1-min files ...` | The bundle lacks the requested experiment. Narrow `EXPERIMENTS`, or rebuild the bundle. |
| `CUDA out of memory` | Switch to a CPU runtime, which is sufficient here. |